# LOB Alpha Research: run everything on Google Colab

**Runtime → Run all.** The notebook clones the repo, restores every result that is already finished, downloads
only the datasets still needed, runs whatever is outstanding, and saves everything to Google Drive.

**One-time setup:**
1. **Runtime → Change runtime type → T4 GPU** (any GPU works; the notebook stops early without one).
2. **Kaggle API key:** on kaggle.com go to *Settings → API → Create New Token*. In Colab, open the 🔑 **Secrets**
   panel on the left, add `KAGGLE_USERNAME` and `KAGGLE_KEY` with the values from that `kaggle.json`, and allow
   notebook access.
3. **Optiver data** (only needed if those experiments rerun): accept the
   [Realized Volatility](https://www.kaggle.com/competitions/optiver-realized-volatility-prediction/rules) and
   [Trading at the Close](https://www.kaggle.com/competitions/optiver-trading-at-the-close/rules) rules with the same account.
4. **Optional `GITHUB_TOKEN` secret:** needed to clone a private repo, and (with write access) to push this
   session's checkpoints to the `run-state` branch at the end.

**What runs by default.** Every experiment has a "done" check. Finished ones are skipped, so with the current
results the run is: Experiment J's DeepLOB part (retrain DeepLOB on the 4 BTC segments, save its predictions and
gradient attribution, then rerun the significance tests including DeepLOB; roughly 30 min on an A100), the tests
and the figures. The LOBSTER experiment (I) is not in this notebook: its data is not on Kaggle, so it runs on the laptop. Change `RUN` / `FORCE` in the settings cell
to choose differently.

**Disconnects.** Keep the tab open. Results and checkpoints go to `MyDrive/ML_Project_2026/artifacts`, and each
long run checkpoints per seed / segment / horizon / fold. After a disconnect just **Run all** again: finished work
is skipped and interrupted runs resume.

## 1. Settings

In [ ]:
REPO = "aaravsin-1/ML_Project_2026"   # GitHub owner/name
BRANCH = "master"
SAVE_TO_DRIVE = True                    # keep artifacts + checkpoints on Google Drive (needed to survive disconnects)
DATA_ON_DRIVE = False                   # raw data on fast local disk (re-downloaded each session)
REQUIRE_GPU = True                      # stop early if no GPU is attached

# Which experiments may run. A step whose results already exist is skipped unless it is listed in FORCE.
RUN = {
    "synthetic":        True,   # Synthetic data, Experiments A-D, 5 seeds            (~15 min)
    "btc":              True,   # Coinbase BTC, Experiments A-D, 4 segments            (~2 h)
    "btc_tradability":  True,   # Experiment E: confidence filtering, cost-aware labels (~1 h)
    "btc_eth":          True,   # Experiment F: does ETH's book help predict BTC?      (~45 min)
    "btc_regimes":      True,   # Experiment G: does the edge survive in any regime?   (~10 min, CPU)
    "btc_market_making": True,  # Experiment H: market making + adverse selection      (~2 min, CPU)
    "deeplob_attribution": True, # Experiment J: DeepLOB predictions + attribution, then significance (~30 min, GPU)
    "fi2010":           True,   # FI-2010 Setup 2, seeds in FI2010_SEEDS               (~1-2 h per seed)
    "fi2010_diagnose":  True,   # Why LightGBM + history fails at k=20                 (~15 min)
    "optiver_close":    True,   # Optiver Trading at the Close                         (~45 min)
    "optiver_rvp":      True,   # Optiver Realized Volatility                          (~45 min)
    "validation":       True,   # Leakage / robustness report (synthetic)              (~5 min)
    "cpp_trained":      False,  # Trained LightGBM -> C++ timing (left for a later rerun on an idle machine)
}
FORCE = set()                   # e.g. {"btc"}: rerun a step even though its results exist. "synthetic" and "btc"
                                # recompute from scratch; the others redo only work without a checkpoint file.
FI2010_SEEDS = [0, 1, 2]

REPO_RESULTS_WIN = True         # results committed in the repo replace older copies on Drive
RUN_TESTS = True
DOWNLOAD_ZIP = True             # download the artifacts as a zip at the end
PUSH_TO_RUN_STATE = True        # push checkpoints to the run-state branch at the end (needs GITHUB_TOKEN with write access)

## 2. Setup: Drive, code, dependencies, credentials

In [ ]:
import os, subprocess, sys, time, shutil
from pathlib import Path

if SAVE_TO_DRIVE or DATA_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive/ML_Project_2026")
os.environ["LOB_ARTIFACTS_DIR"] = str(drive_root / "artifacts" if SAVE_TO_DRIVE else Path("/content/artifacts"))
os.environ["LOB_DATA_DIR"] = str(drive_root / "data" if DATA_ON_DRIVE else Path("/content/data"))
os.environ["PYTHONUNBUFFERED"] = "1"
art = Path(os.environ["LOB_ARTIFACTS_DIR"])
for p in (art, Path(os.environ["LOB_DATA_DIR"])):
    p.mkdir(parents=True, exist_ok=True)

from google.colab import userdata
def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
repo_dir = Path("/content/ML_Project_2026")
if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(repo_dir)], check=True)
os.chdir(repo_dir)
print("Code at commit:", subprocess.run(["git", "log", "-1", "--format=%h %ci %s"], capture_output=True, text=True).stdout.strip())
if "def gradient_x_input" not in Path("src/experiments/exp_j_attribution.py").read_text():
    raise RuntimeError("The cloned code is older than this notebook (it lacks the cuDNN-safe DeepLOB attribution). "
                       "Push the latest code to GitHub first, then Run all again.")

for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    value = secret(name)
    if value:
        os.environ[name] = value
print("Kaggle credentials:", "found" if os.environ.get("KAGGLE_KEY") else "MISSING - add the Colab secrets (see top)")
print("GitHub token:      ", "found" if token else "not set (fine for a public repo; no push at the end)")
print("Artifacts ->", art)
print("Data      ->", os.environ["LOB_DATA_DIR"])

In [ ]:
!pip install -q -r requirements.txt
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif REQUIRE_GPU:
    raise RuntimeError("No GPU attached. Runtime -> Change runtime type -> T4 GPU, then Run all again "
                       "(or set REQUIRE_GPU = False to run on CPU, which is far slower).")
else:
    print("WARNING: running on CPU; the deep-learning steps will be very slow.")

## 3. Restore finished results
Copies the results committed in the repo into the artifacts folder, then anything else from the `run-state`
branch (checkpoints pushed by the laptop or an earlier Colab session). Files that exist only on Drive, such as
checkpoints of an interrupted run, are always kept.

In [ ]:
repo_art = repo_dir / "artifacts"
copied = replaced = 0
for src in repo_art.rglob("*"):
    if not src.is_file():
        continue
    dst = art / src.relative_to(repo_art)
    if dst.exists() and (not REPO_RESULTS_WIN or dst.read_bytes() == src.read_bytes()):
        continue
    replaced += dst.exists()
    copied += not dst.exists()
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
print(f"[+] from the repo: {copied} new file(s), {replaced} replaced")
!python scripts/sync_checkpoints.py --restore

## 4. Plan
Lists each step and whether it will run. A step counts as done when all of its result files exist.

In [ ]:
def fi2010_done_files(seeds):
    files = []
    for s in seeds:
        for k in (10, 20, 50, 100):
            files.append(f"fi2010/setup2_k{k}_e15{'' if s == 0 else f'_s{s}'}.csv")
            files.append(f"fi2010/setup2_k{k}_history_s{s}.csv")
    return files

PY = "python -u -W ignore"
STEPS = {
    "synthetic": dict(cmd=f"{PY} scripts/run_all_experiments.py --source synthetic --seeds 0 1 2 3 4 --epochs 10",
                      data=[], done=["experiment_a_results.csv", "experiment_d_latency_sharpe.csv"]),
    "btc": dict(cmd=f"{PY} scripts/run_all_experiments.py --source btc --segments 4 --epochs 8",
                data=["btc"], done=["btc/experiment_a_results.csv", "btc/experiment_d_latency_sharpe.csv"]),
    "btc_tradability": dict(cmd=f"{PY} -m src.experiments.exp_e_tradability",
                            data=["btc"], done=["btc/tradability/tradability_summary.csv"]),
    "btc_eth": dict(cmd=f"{PY} -m src.experiments.exp_f_cross_asset",
                    data=["btc", "eth"], done=["btc/cross_asset/cross_asset_summary.csv"]),
    "btc_regimes": dict(cmd=f"{PY} -m src.experiments.exp_g_regimes",
                        data=["btc"], done=["btc/regimes/regime_tradability.csv"]),
    "btc_market_making": dict(cmd=f"{PY} -m src.experiments.exp_h_market_making",
                              data=["btc"], done=["btc/market_making/mm_summary.csv"]),
    "deeplob_attribution": dict(cmd=f"{PY} -m src.experiments.exp_j_attribution --deeplob",
                                data=["btc"], done=["btc/attribution/deeplob_attribution.csv"]),
    "fi2010": dict(cmd=f"{PY} -m src.experiments.fi2010_benchmark --epochs 15 --seeds " + " ".join(map(str, FI2010_SEEDS)),
                   data=["fi2010"], done=fi2010_done_files(FI2010_SEEDS)),
    "fi2010_diagnose": dict(cmd=f"{PY} scripts/diagnose_fi2010_lightgbm.py",
                            data=["fi2010"], done=["fi2010/diagnose_lightgbm_history.csv"]),
    "optiver_close": dict(cmd=f"{PY} -m src.experiments.optiver_close",
                          data=["optiver_close"], done=["optiver_close/close_holdout_results.csv"]),
    "optiver_rvp": dict(cmd=f"{PY} -m src.experiments.optiver_rvp",
                        data=["optiver_rvp"], done=["optiver_rvp/rvp_summary.csv"]),
    "validation": dict(cmd=f"{PY} scripts/verify_robustness_and_leakage.py --seed 42 --events 60000",
                       data=[], done=["overfitting_verification_report.csv"]),
    "cpp_trained": dict(cmd=f"{PY} scripts/benchmark_trained_model.py",
                        data=["btc"], done=["cpp_trained_model_benchmark.csv"]),
}

def is_done(name):
    return all((art / f).exists() for f in STEPS[name]["done"])

def will_run(name):
    return RUN.get(name, False) and (name in FORCE or not is_done(name))

for name in STEPS:
    if not RUN.get(name, False):
        state = "off"
    elif name in FORCE:
        state = "RUN (forced)"
    elif is_done(name):
        state = "done, skip"
    else:
        missing = [f for f in STEPS[name]["done"] if not (art / f).exists()]
        state = f"RUN ({len(missing)} result file(s) missing)"
    print(f"  {name:<17} {state}")

## 5. Download the datasets the plan needs

In [ ]:
needed = sorted({d for name in STEPS if will_run(name) for d in STEPS[name]["data"]})
FAILED_DATA = set()
if not needed:
    print("No datasets needed: every selected step is already done.")
elif not os.environ.get("KAGGLE_KEY"):
    FAILED_DATA = set(needed)
    print("[!] No Kaggle credentials, so nothing can be downloaded. Add the KAGGLE_USERNAME / KAGGLE_KEY secrets "
          "(see the top), then Run all again. Steps needing", ", ".join(needed), "will be skipped.")
else:
    for d in needed:
        if subprocess.run(["python", "scripts/fetch_datasets.py", d]).returncode != 0:
            FAILED_DATA.add(d)
    print("\nDatasets:", ", ".join(f"{d}={'FAILED' if d in FAILED_DATA else 'ok'}" for d in needed))
    if FAILED_DATA:
        print("[!] Steps that need", ", ".join(sorted(FAILED_DATA)), "will be skipped; the messages above say why.")

## 6. Run
Each cell runs one step. A failing step is reported and the rest still run; the summary at the end lists what
happened. Re-running a cell resumes from its checkpoints.

In [ ]:
STATUS = {}

def run_step(name, cmd=None, force_run=False):
    """Runs one step as a subprocess, streaming its output; never raises, so Run all continues."""
    if not force_run and not will_run(name):
        STATUS[name] = "off" if not RUN.get(name, False) else "done earlier"
        print(f"[=] {name}: {STATUS[name]}, skipped")
        return
    lacking = sorted(set(STEPS.get(name, {}).get("data", [])) & FAILED_DATA)
    if lacking:
        STATUS[name] = f"SKIPPED: download of {', '.join(lacking)} failed (see section 5)"
        print(f"[!] {name}: {STATUS[name]}")
        return
    cmd = cmd or STEPS[name]["cmd"]
    if name in FORCE and name in ("synthetic", "btc"):
        cmd += " --fresh"  # these two cache whole sessions; recompute them instead of reloading
    print(f"[>] {name}: {cmd}\n", flush=True)
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code_ = proc.wait()
    minutes = (time.time() - t0) / 60
    STATUS[name] = f"ok ({minutes:.0f} min)" if code_ == 0 else f"FAILED, exit code {code_} ({minutes:.0f} min)"
    print(f"\n[{'+' if code_ == 0 else '!'}] {name}: {STATUS[name]}")

In [ ]:
# 6a. Tests (the torch-dependent tests can only run here, not on the laptop)
if RUN_TESTS:
    run_step("tests", "python -m pytest tests -q", force_run=True)

In [ ]:
# 6b. FI-2010 Setup 2 (checkpointed per seed and horizon)
run_step("fi2010")

In [ ]:
# 6c. FI-2010 LightGBM + history diagnostic
run_step("fi2010_diagnose")

In [ ]:
# 6d. Synthetic data, Experiments A-D (checkpointed per seed)
run_step("synthetic")

In [ ]:
# 6e. Coinbase BTC, Experiments A-D (checkpointed per segment)
run_step("btc")

In [ ]:
# 6f. Experiment E: BTC tradability (checkpointed per segment)
run_step("btc_tradability")

In [ ]:
# 6g. Experiment F: ETH -> BTC (checkpointed per segment)
run_step("btc_eth")

In [ ]:
# 6g2. Experiment G: regimes (CPU; usually already done on the laptop)
run_step("btc_regimes")

In [ ]:
# 6g3. Experiment H: market making (CPU; usually already done on the laptop)
run_step("btc_market_making")

In [ ]:
# 6g4. Experiment J: DeepLOB predictions and attribution (GPU), then significance tests
run_step("deeplob_attribution")

In [ ]:
# 6h. Optiver Trading at the Close (checkpointed per stage)
run_step("optiver_close")

In [ ]:
# 6i. Optiver Realized Volatility (checkpointed per fold)
run_step("optiver_rvp")

In [ ]:
# 6j. Leakage / robustness report
run_step("validation")

In [ ]:
# 6k. Trained LightGBM in C++ (off by default)
run_step("cpp_trained")

In [ ]:
# 6l. Real-data figures (always rebuilt from the saved CSVs)
run_step("figures", f"{PY} scripts/make_real_data_figures.py", force_run=True)

## 7. Summary and results

In [ ]:
import pandas as pd
print("Step status:")
for name, state in STATUS.items():
    print(f"  {name:<17} {state}")

tables = {
    "FI-2010 Setup 2 (mean / std / count over seeds)": art / "fi2010" / "fi2010_setup2_summary.csv",
    "FI-2010 LightGBM + history diagnostic": art / "fi2010" / "diagnose_lightgbm_history.csv",
    "BTC, Experiment A": art / "btc" / "experiment_a_results.csv",
    "BTC, Experiment D": art / "btc" / "experiment_d_execution_strategies.csv",
    "BTC, Experiment F (ETH -> BTC)": art / "btc" / "cross_asset" / "cross_asset_summary.csv",
    "Optiver Close holdout": art / "optiver_close" / "close_holdout_results.csv",
    "Optiver RVP forward CV": art / "optiver_rvp" / "rvp_summary.csv",
    "Significance vs LightGBM (BTC)": art / "btc" / "attribution" / "significance_vs_lightgbm.csv",
    "DeepLOB attribution (BTC)": art / "btc" / "attribution" / "deeplob_attribution.csv",
    "Synthetic, Experiment A": art / "experiment_a_results.csv",
}
pd.set_option("display.width", 200)
for title, path in tables.items():
    print(f"\n=== {title} ===")
    print(pd.read_csv(path).round(4).to_string(index=False) if path.exists() else "(not run yet)")

## 8. Take the results home
Downloads a zip of the artifacts folder (per-run `sessions/` pickles left out). Unzip it into the repo's
`artifacts/` folder on the laptop. Everything also stays on Drive.

In [ ]:
if DOWNLOAD_ZIP:
    try:
        from google.colab import files
        staging = Path("/content/artifacts_export")
        shutil.rmtree(staging, ignore_errors=True)
        shutil.copytree(art, staging, ignore=shutil.ignore_patterns("sessions", "*.pkl"))
        zip_path = shutil.make_archive("/content/artifacts", "zip", staging)
        files.download(zip_path)
        print("[+] download started:", zip_path)
    except Exception as e:
        print("[!] zip download failed:", e, "- the results are still on Drive at", art)

## 9. Push checkpoints to the `run-state` branch
Lets the laptop or another Colab session resume from this one with `python scripts/sync_checkpoints.py --restore`.
Skipped without a `GITHUB_TOKEN` secret that has write access.

In [ ]:
if PUSH_TO_RUN_STATE and token:
    subprocess.run("git config --global user.email 'colab@example.com' && git config --global user.name 'Colab runner'", shell=True)
    subprocess.run(["python", "scripts/sync_checkpoints.py"])
else:
    print("Skipped (no GITHUB_TOKEN secret, or PUSH_TO_RUN_STATE is False).")